In [ ]:
import json
import re
import time
from anthropic import APIError
from datetime import datetime
from langchain.schema import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_xai import ChatXAI
from langchain_anthropic import ChatAnthropic
from langchain_core.tracers.context import tracing_v2_enabled
from langsmith import traceable
import uuid
from langsmith import Client
from langsmith.run_helpers import get_current_run_tree
from langchain_community.callbacks import get_openai_callback
import pandas as pd
import ast
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from IPython.display import Image, display
from __future__ import annotations
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from typing import Annotated
from operator import itemgetter  # For taking the last value

class State(TypedDict):
    messages: Annotated[list, add_messages]
    phase: Annotated[str, lambda _, x: x]
    user_profile: dict
    step_idx: int
    same_step_turns: int
    selected_recipe: dict | None
    chef_agent: 'ChefAgent'
    trainee_agent: 'TraineeAgent'
    user_query: str | None
    clarified_topics: list[str]
    active_subgraph: Annotated[str | None, lambda _, x: x]
    step_subgraphs: Annotated[dict, lambda _, x: x]
    hsm: Annotated['DynamicHSM', lambda _, x: x]  # <-- ADD THIS LINE


class DynamicHSM:
    def __init__(self):
        self.subgraphs = {}  # {subgraph_id: CompiledStateGraph}

    def create_clarification_subgraph(self, step_idx):
        print(f"\n=== Creating clarification subgraph for step {step_idx} ===")
        subgraph_builder = StateGraph(State)
        subgraph_builder.add_node("clarification_chef", clarification_chef_node)
        subgraph_builder.add_node("clarification_trainee", clarification_trainee_node)
        
        # Add entry point
        subgraph_builder.add_edge(START, "clarification_chef")  # <-- THIS WAS MISSING
        subgraph_builder.add_edge("clarification_chef", "clarification_trainee")
        
        subgraph_builder.add_conditional_edges(
            "clarification_trainee",
            lambda s: "clarification_chef" if not s.get("clarified") else END,
            {"clarification_chef": "clarification_chef", END: END}
        )
        compiled_subgraph = subgraph_builder.compile()
        self.subgraphs[f"clarification_{step_idx}"] = compiled_subgraph
        return compiled_subgraph

    



def user_select_recipe_node(state: State):
    # Print options for the user
    last_ai_msg = state["messages"][-1].content
    print(f"\nChef: {last_ai_msg}")
    user_choice = input("Which recipe would you like to cook? ").strip()
    recipe = get_recipe_by_name(user_choice, recipes_df)
    allergies = state["user_profile"].get("allergies", [])
    if recipe is not None:
        state["selected_recipe"] = recipe
        if contains_allergen(recipe['Cleaned_Ingredients'], allergies):
            state["phase"] = "allergy_warning"
        else:
            state["phase"] = "ingredient_check"
    else:
        state["phase"] = "recipe_selection"
    return state

def allergy_warning_node(state: State):
    recipe_title = state["selected_recipe"]['Title']
    allergen_list = state["user_profile"].get("allergies", [])
    warning_msg = (
        f"WARNING: The recipe '{recipe_title}' contains your allergen(s): {', '.join(allergen_list)}.\n"
        "Are you absolutely sure you want to proceed? (yes/no)"
    )
    print(f"\nChef: {warning_msg}")
    user_input = input("Proceed with this recipe? (yes/no): ").strip().lower()
    if user_input == "yes":
        state["phase"] = "ingredient_check"
        # Keep selected_recipe intact!
    else:
        state["selected_recipe"] = None  # Clear recipe
        state["phase"] = "recipe_selection"
    return state



def recipe_selection_node(state: State):
    # Always use the latest user_query
    intent = state["chef_agent"].detect_intent(state["user_query"])
    
    # Handle different intents
    if intent.intent_type == "specific_recipe":
        recipe = get_recipe_by_name(intent.target_recipe, recipes_df)
    elif intent.intent_type == "similar_recipes":
        similar = get_similar_recipes(intent.target_recipe, recipes_df)
        # Let user pick from similar recipes
        state["messages"].append(AIMessage(content=f"Similar recipes: {similar['Title'].tolist()}"))
        return state  # Return early after handling
    elif intent.intent_type == "ingredient_search":
        matches = get_recipes_by_ingredients(intent.ingredients, recipes_df)
        # Let user pick from matches
        state["messages"].append(AIMessage(content=f"Matching recipes: {matches['Title'].tolist()}"))
        return state  # Return early after handling
    else:
        state["messages"].append(AIMessage(content="Please clarify your request"))
        return state

def introduction_node(state: State):
    state["messages"].append(AIMessage(content=state["chef_agent"].system_message.content))
    state["phase"] = "recipe_selection"
    return state

from langgraph.types import interrupt

def ingredient_check_node(state: State):
    recipe = state["selected_recipe"]
    ingredients = ast.literal_eval(recipe['Cleaned_Ingredients'])
    msg = AIMessage(
        content=f"Required ingredients:\n" + "\n".join(f"- {i}" for i in ingredients) +
                "\nDo you have all ingredients? (yes/no)"
    )
    state["messages"].append(msg)

    # Wait for user input (human-in-the-loop)
    user_input = input("Trainee: ").strip().lower()
    state["messages"].append(HumanMessage(content=user_input))
    if user_input in ["yes", "y"]:
        state["phase"] = "chef"  # <-- Ensure this is set
    else:
        # Prompt for a new query
        new_query = input("What would you like to cook instead? ")
        state["user_query"] = new_query
        state["selected_recipe"] = None
        state["phase"] = "recipe_selection"
    return state



def chef_node(state: State):
    # Check if recipe is selected
    if state["selected_recipe"] is None:
        state["messages"].append(AIMessage(content="No recipe selected. Let's pick a new one."))
        state["phase"] = "recipe_selection"
        return state  # Early return if no recipe

    # Parse steps and get current step index
    steps = parse_steps(state["selected_recipe"]['Instructions'])
    idx = state.get("step_idx", 0)

    # If all steps are done, transition to done phase
    if idx >= len(steps):
        state["phase"] = "done"
        state["messages"].append(AIMessage(content="Recipe complete! Well done."))
        return state

    step_text = steps[idx]
    complexity = classify_step_complexity(step_text)

    # Only create a clarification subgraph if not already present for this step
    if f"clarification_{idx}" not in state["step_subgraphs"]:
        hsm = state["hsm"]  # <-- get the HSM instance from the state
        subgraph = hsm.create_clarification_subgraph(idx)
        state["step_subgraphs"][f"clarification_{idx}"] = subgraph

    # Present the step to the user
    state["messages"].append(
        AIMessage(content=f"Step {idx+1}: {step_text} (Complexity: {complexity})")
    )

    # Set phase for next expected user action (e.g., waiting for clarification or 'next')
    state["phase"] = "trainee"

    return state




def trainee_node(state: State):
    last_chef_msg = [m.content for m in state["messages"] if isinstance(m, AIMessage)][-1]
    num_steps = len(parse_steps(state["selected_recipe"]['Instructions']))
    trainee_response = state["trainee_agent"].generate_response(last_chef_msg, num_steps)
    state["messages"].append(HumanMessage(content=trainee_response))

    if is_clarification_request(trainee_response):
        # Create subgraph only when needed
        hsm = DynamicHSM()
        subgraph = hsm.create_clarification_subgraph(state["step_idx"])
        state["step_subgraphs"][state["step_idx"]] = subgraph
        state["active_subgraph"] = f"clarification_{state['step_idx']}"
        return state

    # 2. Substitution request
    elif is_substitution_request(trainee_response):
        hsm = DynamicHSM()
        subgraph = hsm.create_substitution_subgraph(state["step_idx"])
        state["step_subgraphs"][state["step_idx"]] = subgraph
        state["active_subgraph"] = f"substitution_{state['step_idx']}"
        state["phase"] = "substitution_subgraph"
        return state

    # 3. Normal step processing
    else:
        # Example: track clarified topics
        if "kosher salt" in trainee_response:
            if "kosher salt" not in state.get("clarified_topics", []):
                state["clarified_topics"].append("kosher salt")

        # Step advancement
        if "next" in trainee_response.lower():
            state["step_idx"] += 1  # <-- THIS WAS MISSING
            state["same_step_turns"] = 0
            state["clarified_topics"] = []
            state["phase"] = "chef"
        else:
            state["same_step_turns"] = state.get("same_step_turns", 0) + 1
            if state["same_step_turns"] >= 3:
                print("ChefAI: Let's move on to the next step to keep cooking moving forward!")
                state["step_idx"] += 1
                state["same_step_turns"] = 0
                state["clarified_topics"] = []
                state["phase"] = "chef_turn"
            else:
                state["phase"] = "chef_turn"
        return state





def build_cooking_graph():
    graph_builder = StateGraph(State)
    
    # Add nodes (including the missing one)
    graph_builder.add_node("introduction", introduction_node)
    graph_builder.add_node("recipe_selection", recipe_selection_node)
    graph_builder.add_node("user_select_recipe", user_select_recipe_node)  # THIS WAS MISSING
    graph_builder.add_node("ingredient_check", ingredient_check_node)
    graph_builder.add_node("chef", chef_node)
    graph_builder.add_node("trainee", trainee_node)
    graph_builder.add_node("allergy_warning", allergy_warning_node)
    
    
    # Define edges
    graph_builder.add_edge(START, "introduction")
    graph_builder.add_edge("introduction", "recipe_selection")
    graph_builder.add_edge("recipe_selection", "user_select_recipe")
    graph_builder.add_conditional_edges(
        "user_select_recipe",
        lambda s: (
            # First check if we need to warn about allergies
            "allergy_warning" if s.get("phase") == "allergy_warning"
            # Then check if recipe was selected
            else "ingredient_check" if s["selected_recipe"] is not None
            # Fallback to recipe selection
            else "recipe_selection"
        ),
        {
            "allergy_warning": "allergy_warning",
            "ingredient_check": "ingredient_check", 
            "recipe_selection": "recipe_selection"
        }
    )
    graph_builder.add_conditional_edges(
    "ingredient_check",
    lambda s: "chef" if s["selected_recipe"] is not None else "recipe_selection",
    {"chef": "chef", "recipe_selection": "recipe_selection"}
)



# From allergy_warning, go to ingredient_check or recipe_selection
    graph_builder.add_conditional_edges(
        "allergy_warning",
        lambda s: "ingredient_check" if s["phase"] == "ingredient_check" else "recipe_selection",
        {
            "ingredient_check": "ingredient_check",
            "recipe_selection": "recipe_selection"
        }
    )

    graph_builder.add_edge("ingredient_check", "chef")
    graph_builder.add_edge("chef", "trainee")
    graph_builder.add_edge("trainee", "chef")
    
    graph_builder.add_conditional_edges(
        "chef",
        lambda s: END if s.get("phase") == "done" else "trainee",
        {END: END, "trainee": "trainee"}
    )
    
    return graph_builder.compile()



class IntentDetection(BaseModel):
    """Identify user intent and extract key details"""
    intent_type: str = Field(..., description="Type of request: 'specific_recipe', 'similar_recipes', 'ingredient_search'")
    target_recipe: str | None = Field(None, description="Recipe name if specified")
    ingredients: list[str] | None = Field(None, description="List of ingredients if provided")
recipes_df = pd.read_csv("13k-recipes.csv")

def classify_step_complexity(step_text):
    # Simple heuristic: long steps or those with multiple actions are 'complex'
    if len(step_text.split()) > 30:
        return "complex"
    if any(word in step_text.lower() for word in ["meanwhile", "until", "while", "simultaneously", "at the same time", "then"]):
        return "complex"
    # You can add more rules as needed
    return "simple"

def clarification_chef_node(state: State):
    last_trainee_msg = [m.content for m in state["messages"] if isinstance(m, HumanMessage)][-1]
    response = state["chef_agent"].respond(
        state["messages"], 
        f"Provide detailed clarification for: {last_trainee_msg}"
    )
    state["messages"].append(AIMessage(content=response))
    return state

def clarification_trainee_node(state: State):
    user_input = input("Trainee (ask follow-up or 'next'): ").strip().lower()
    state["messages"].append(HumanMessage(content=user_input))
    state["clarified"] = (user_input == "next")
    return state


client = Client()
runs = client.list_runs(project_name="run-traces")

# for run in runs:
#     print("Run name:", run.name)
#     print("Start time:", run.start_time)
#     print("End time:", run.end_time)
#     print("Duration (seconds):", (run.end_time - run.start_time).total_seconds())

class ChefAgent:
    def __init__(self, job_id, trainee_experience_log=None):
        # self.llm = ChatXAI(model="",
        #                     xai_api_key=os.environ["XAI_API_KEY"]
        # )
        self.llm = ChatOpenAI(model="gpt-4.1-mini", openai_api_key=os.environ["OPENAI_API_KEY"])
        self.structured_llm = ChatOpenAI(model="gpt-4.1-mini", openai_api_key=os.environ["OPENAI_API_KEY"]).with_structured_output(IntentDetection)
        self.job_id = job_id
        self.token_cost_log = []    
        self.trainee_experience_log = trainee_experience_log
        # self.llm = ChatAnthropic(model="claude-3-5-sonnet-20240620",temperature=0)
        self.system_message = self.build_system_message()

    def detect_intent(self, user_query: str) -> IntentDetection:
        prompt = f"""Analyze this cooking-related query:
        {user_query}

        Classify the intent:
        - 'specific_recipe' if asking for a particular dish
        - 'similar_recipes' if requesting variations
        - 'ingredient_search' if listing ingredients they have
        
        Extract recipe names or ingredients as needed."""
        
        return self.structured_llm.invoke(prompt)

    def build_system_message(self):
        experience_level = self.trainee_experience_log.get('experience_level', 'unknown') if self.trainee_experience_log else 'unknown'
        allergies = self.trainee_experience_log.get('allergies', []) if self.trainee_experience_log else []
        preferred_cuisine = self.trainee_experience_log.get('preferred_cuisine', 'any') if self.trainee_experience_log else 'any'
        notes = self.trainee_experience_log.get('notes', '') if self.trainee_experience_log else ''

        allergy_str = ", ".join(allergies) if allergies else "none"

        experience_info = f"""
    User Profile:
    - Experience level: {experience_level}
    - Allergies: {allergy_str}
    - Preferred cuisine: {preferred_cuisine}
    - Notes: {notes}

    IMPORTANT INSTRUCTIONS:
    - NEVER suggest or proceed with any recipe or step that contains any of the user's allergens: {allergy_str}.
    - If the user requests a recipe with an allergen, gently warn them and suggest safe alternatives.
    - ALWAYS adapt your explanations to the user's experience level. For beginners, explain techniques, tools, and terminology in simple terms, and offer encouragement.
    - If the user has never performed a technique before (see notes), provide extra explanation and support when that technique arises.
    - Before each step, check if the user is comfortable and ready to proceed.
    """

        base_instructions = """
    Please follow these instructions carefully:

    1. Introduction:
    - Introduce yourself as ChefAI.
    - Briefly explain that you're here to help with recipe preparation.

    2. Recipe Confirmation:
    - Confirm the recipe name with the user.

    3. Ingredient Recall:
    - List all the ingredients required for the recipe.
    - Ask the user if they have all the ingredients ready.

    4. Step-by-Step Guidance:
    - Provide instructions one step at a time.
    - After each step, wait for the user to say "next" or ask a question before proceeding.
    - If the user asks a question, answer it thoroughly before continuing with the recipe steps.

    5. Completion:
    - When all steps are complete, congratulate the user and ask if they need any final advice.

    Before each interaction with the user, wrap your analysis in <recipe_analysis> tags to process the recipe, organize your thoughts, and prepare your response. In this analysis:
    - Break down the recipe into key components: ingredients, equipment needed, and cooking techniques.
    - Identify potential challenges or areas where users might need extra guidance.
    - Plan out how to present each step in a clear and concise manner.
    This will help you provide accurate and helpful guidance without revealing all instructions at once.

    Example interaction format:

    ChefAI: [Introduction and recipe confirmation]
    User: [Response]
    ChefAI: [List ingredients and ask if ready]
    User: [Response]
    ChefAI: [Provide first step]
    User: next
    ChefAI: [Provide next step]
    User: [Question]
    ChefAI: [Answer question]
    User: next
    ... (continue until recipe is complete)

    Remember to maintain a friendly and encouraging tone throughout the interaction. Begin your first response now by introducing yourself and confirming the recipe.
    """
        return SystemMessage(content=experience_info + base_instructions)


    @traceable(run_type="chain", name="Chef Response", metadata={"role": "chef"})
    def respond(self, conversation, prompt):
        run = get_current_run_tree()
        if run:
            run.metadata["job_id"] = self.job_id
            run.metadata["run_id"] = str(run.id)
        messages = [self.system_message] + conversation + [HumanMessage(content=prompt)]
        max_retries = 5
        base_delay = 1

        for attempt in range(max_retries):
            try:
                with get_openai_callback() as cb:
                    response = self.llm.invoke(messages).content.strip()
                # Log token and cost info
                self.token_cost_log.append({
                    "prompt_tokens": cb.prompt_tokens,
                    "completion_tokens": cb.completion_tokens,
                    "total_tokens": cb.total_tokens,
                    "cost": cb.total_cost,
                    "timestamp": datetime.now().isoformat(),
                    "prompt": prompt
                })
                update_running_totals(self.token_cost_log)
                print(f"ChefAgent LLM call: {cb.total_tokens} tokens, ${cb.total_cost:.5f}")
                return response
            except APIError as e:
                if "overloaded_error" in str(e):
                    delay = base_delay * (2 ** attempt)
                    print(f"OverloadedError: Retrying in {delay}s (attempt {attempt+1}/{max_retries})")
                    time.sleep(delay)
                else:
                    raise
        raise Exception("Max retries exceeded")


class TraineeAgent:
    def __init__(self, job_id, trainee_experience_log=None, conversation_mode="mixed"):
        self.llm = ChatOpenAI(model="gpt-4.1-mini", openai_api_key=os.environ["OPENAI_API_KEY"])
        self.job_id = job_id
        self.current_step = 0
        self.recipe_data = None
        self.conversation_mode = conversation_mode
        self.token_cost_log = []
        self.trainee_experience_log = trainee_experience_log
        
    @traceable(run_type="chain", name="Trainee Response", metadata={"role": "trainee"})
    def generate_response(self, chef_message, num_steps):
        experience_level = self.trainee_experience_log.get("experience_level", "beginner").lower()
        notes = self.trainee_experience_log.get("notes", "")

        # Adjust question-asking behavior based on experience
        if experience_level == "advanced":
            question_instruction = (
                "ONLY ask a question if this step is ambiguous or unusually challenging for an expert cook. "
                "If everything is clear, say 'next'. Do NOT ask about basic techniques or substitutions."
            )
        elif experience_level == "intermediate":
            question_instruction = (
                "Ask a question if you are unsure about a technique or ingredient. "
                "Otherwise, say 'next'."
            )
        else:  # beginner
            question_instruction = (
                "If you have any doubt about the technique, ingredient, or process in this step, ask ONE SHORT, direct question. "
                "Otherwise, say 'next'."
            )

        # Optionally, use notes for further customization (e.g., never roasted a chicken)
        if notes and "never" in notes.lower():
            question_instruction += (
                f" You have noted: {notes}. If this step involves something you have never done, ask for extra explanation."
            )

        prompt = f"""You're following a recipe with {num_steps} steps. Current step: {self.current_step+1}.
    Last instruction: {chef_message}
    {question_instruction}
    """
        with get_openai_callback() as cb:
            response = self.llm.invoke([HumanMessage(content=prompt)])
            self.token_cost_log.append({
                "prompt_tokens": cb.prompt_tokens,
                "completion_tokens": cb.completion_tokens,
                "total_tokens": cb.total_tokens,
                "cost": cb.total_cost,
                "timestamp": datetime.now().isoformat(),
                "prompt": prompt
            })
            update_running_totals(self.token_cost_log)
            print(f"TraineeAgent LLM call: {cb.total_tokens} tokens, ${cb.total_cost:.5f}")
            return response.content.strip().lower()


def parse_steps(recipe_text):
    steps = [s.strip() for s in re.split(r'\n{2,}|\n', recipe_text) if s.strip()]
    return steps

def update_running_totals(log_list):
    total_tokens = 0
    total_cost = 0
    for entry in log_list:
        total_tokens += entry["total_tokens"]
        total_cost += entry["cost"]
        entry["cumulative_tokens"] = total_tokens
        entry["cumulative_cost"] = total_cost

def append_with_accrual(conversation, message, chef, trainee, combined_accrual):
    conversation.append(message)
    chef_cum = chef.token_cost_log[-1] if chef.token_cost_log else {"cumulative_tokens": 0, "cumulative_cost": 0}
    trainee_cum = trainee.token_cost_log[-1] if trainee.token_cost_log else {"cumulative_tokens": 0, "cumulative_cost": 0}
    combined_accrual.append({
        "chef_cumulative_tokens": chef_cum["cumulative_tokens"],
        "chef_cumulative_cost": chef_cum["cumulative_cost"],
        "trainee_cumulative_tokens": trainee_cum["cumulative_tokens"],
        "trainee_cumulative_cost": trainee_cum["cumulative_cost"],
        "overall_cumulative_tokens": chef_cum["cumulative_tokens"] + trainee_cum["cumulative_tokens"],
        "overall_cumulative_cost": chef_cum["cumulative_cost"] + trainee_cum["cumulative_cost"]
    })

# # Example experience log
# trainee_experience_log = {
#     "experience_level": "beginner",  # Options: 'beginner', 'intermediate', 'advanced'
#     "preferred_cuisine": "Italian",
#     "allergies": ["nuts"]
# }


def get_recipe_by_name(recipe_name, df):
    matches = df[df['Title'].str.lower().str.contains(recipe_name.lower(), na=False)]
    if not matches.empty:
        return matches.iloc[0]  # Return the first match as a Series (row)
    else:
        return None

def get_similar_recipes(dish_name, df):
    # Handle NaN in Title column
    return df[df['Title'].str.lower().str.contains(dish_name.lower(), na=False)].head(5)

def get_recipes_by_ingredients(ingredients, df):
    mask = df['Cleaned_Ingredients'].apply(
        # Handle NaN in Cleaned_Ingredients
        lambda x: all(ing.lower() in str(x).lower() for ing in ingredients) if pd.notnull(x) else False
    )
    return df[mask].head(5)

def contains_allergen(ingredients, allergies):
    try:
        ingredients_list = ast.literal_eval(ingredients)
    except Exception:
        ingredients_list = [ingredients]
    return any(allergen.lower() in " ".join(ingredients_list).lower() for allergen in allergies)

def visualize_graph(graph):
    """Visualize the LangGraph structure using built-in methods"""
    display(Image(graph.get_graph().draw_mermaid_png()))
    print(graph.get_graph().draw_mermaid())
    return graph.get_graph().draw_mermaid()

    
def create_substitution_subgraph(self, step_idx):
    subgraph = StateGraph(State)
    subgraph.add_node("substitution_chef", substitution_chef_node)
    subgraph.add_node("substitution_trainee", substitution_trainee_node)
    subgraph.add_edge("substitution_chef", "substitution_trainee")
    subgraph.add_conditional_edges(
        "substitution_trainee",
        lambda s: "substitution_chef" if not s.get("substituted") else "done",
        {"substitution_chef": "substitution_chef", "done": END}
    )
    self.subgraphs[f"substitution_{step_idx}"] = subgraph
    return subgraph

def is_substitution_request(text: str) -> bool:
    """Detect if user is asking for ingredient substitution"""
    sub_keywords = [
        "substitute", "replace", "instead of", 
        "alternative", "don't have", "without"
    ]
    return any(kw in text.lower() for kw in sub_keywords)


    # Initialize state
def automated_cooking_session(job_id, trainee_experience_log=None):
    chef = ChefAgent(job_id, trainee_experience_log)
    trainee = TraineeAgent(job_id, trainee_experience_log=trainee_experience_log)
    conversation = []
    combined_accrual = []
    interaction_log = []

    
    # Get user query first
    user_query = input("What would you like to cook? ")
    
    initial_state = {
        "messages": [],
        "phase": "introduction",
        "user_profile": trainee_experience_log,
        "step_idx": 0,
        "same_step_turns": 0,
        "selected_recipe": None,
        "chef_agent": chef,
        "trainee_agent": trainee,
        "user_query": user_query,
        "hsm": DynamicHSM(),
        "clarified_topics": []  # Explicitly initialize as empty list
    }
    
    # Build and compile graph
    graph = build_cooking_graph()
    
    # Increase recursion limit and add debugging
    config = {"recursion_limit": 100}  # From 25 to 100
    
    # Print graph structure
    print("\n" + visualize_graph(graph) + "\n")
    
    # Run conversation loop with debugging
    for output in graph.stream(initial_state, config=config, stream_mode="values"):
        # Print main graph messages
        chef_msgs = [msg for msg in output.get("messages", []) if isinstance(msg, AIMessage)]
        if chef_msgs:
            print(f"\nChef: {chef_msgs[-1].content}")

        trainee_msgs = [msg for msg in output.get("messages", []) if isinstance(msg, HumanMessage)]
        if trainee_msgs:
            print(f"Trainee: {trainee_msgs[-1].content}")

        # --- DYNAMIC SUBGRAPH EXECUTION ---
        if output.get("active_subgraph"):
            print(f"\nENTERING SUBGRAPH: {output['active_subgraph']}")
            subgraph_id = output["active_subgraph"]
            step_idx = output["step_idx"]
            compiled_subgraph = output["step_subgraphs"][step_idx]  # Now a CompiledStateGraph
            sub_state = output.copy()
            
            # Run the COMPILED subgraph
            for sub_output in compiled_subgraph.stream(sub_state):
                # Print subgraph messages
                chef_msgs = [msg for msg in sub_output.get("messages", []) if isinstance(msg, AIMessage)]
                if chef_msgs:
                    print(f"\nChef (Clarification): {chef_msgs[-1].content}")
                trainee_msgs = [msg for msg in sub_output.get("messages", []) if isinstance(msg, HumanMessage)]
                if trainee_msgs:
                    print(f"Trainee: {trainee_msgs[-1].content}")

                # Check for subgraph completion (clarification or substitution resolved)
                if sub_output.get("clarified", False):
                    output["active_subgraph"] = None
                    output["clarified"] = False  # Reset for next use
                    output["phase"] = "chef"     # Return to main step
                    break
            continue  # Resume main graph after sub-dialogue

        # Check for main graph completion
        if output.get("phase") == "done":
            print("\n=== Conversation Completed ===")
            break





    print("ChefAgent system prompt:\n", chef.system_message.content)
    chef_greeting = chef.respond(conversation, "Introduce yourself as ChefAI, your friendly cooking assistant. Ask the user what they'd like to do: find a specific recipe, get similar recipes, or find recipes by ingredients.")
    print("Chef:", chef_greeting)
    append_with_accrual(conversation, AIMessage(content=chef_greeting), chef, trainee, combined_accrual)

    # User selects query type
    user_query = input("What would you like to cook? ")
    intent = chef.detect_intent(user_query)  # Now returns IntentDetection
    
    # Access fields directly
    print(f"Detected intent: {intent.intent_type}")
    print(f"Target recipe: {intent.target_recipe}")
    print(f"Ingredients: {intent.ingredients}")

    if intent.intent_type == "specific_recipe":
        selected_recipe = get_recipe_by_name(intent.target_recipe, recipes_df)
        if selected_recipe is None:
            print("No recipe found.")
            return
    elif intent.intent_type == "similar_recipes":
        similar = get_similar_recipes(intent.target_recipe, recipes_df)
        if similar.empty:
            print("No similar recipes found.")
            return
        print("Similar recipes found:")
        print(similar['Title'].to_list())
        user_choice = input("Which recipe do you want? ")
        selected_recipe = get_recipe_by_name(user_choice, recipes_df)
        if selected_recipe is None:
            print("No recipe found.")
            return
    elif intent.intent_type == "ingredient_search":
        matches = get_recipes_by_ingredients(intent.ingredients, recipes_df)
        if matches.empty:
            print("No recipes found for those ingredients.")
            return
        print("Recipes you can make:")
        print(matches['Title'].to_list())
        user_choice = input("Which recipe do you want? ")
        selected_recipe = get_recipe_by_name(user_choice, recipes_df)
        if selected_recipe is None:
            print("No recipe found.")
            return
    else:
        print("Please clarify your request")
        return


    # Now proceed with your step-by-step logic using selected_recipe
    initial_state["selected_recipe"] = selected_recipe
    recipe_text = selected_recipe['Instructions']
    steps = parse_steps(recipe_text)
    ingredients = selected_recipe['Cleaned_Ingredients']

    try:
        ingredients_list = ast.literal_eval(ingredients)
        if not isinstance(ingredients_list, list):
            ingredients_list = [ingredients]  # fallback: treat as single string
    except Exception:
        ingredients_list = [ingredients]  # fallback: treat as single string

    # 2. Chef lists ingredients and asks if trainee is ready
    ingredients_message = (
        f"Here are the ingredients you'll need for '{selected_recipe['Title']}':\n"
        + "\n".join(f"- {item}" for item in ingredients_list)
        + "\nDo you have all these ingredients ready? (yes/no)"
    )
    print("\nChefAI:", ingredients_message)
    append_with_accrual(conversation, AIMessage(content=ingredients_message), chef, trainee, combined_accrual)

    # 3. Wait for trainee confirmation before proceeding
    trainee_response = input("\nTrainee: ").strip().lower()
    while trainee_response not in ["yes", "y"]:
        print("ChefAI: Please gather all the ingredients before we start. Let me know when you're ready!")
        append_with_accrual(conversation, AIMessage(content="Please gather all the ingredients before we start. Let me know when you're ready!"), chef, trainee, combined_accrual)
        trainee_response = input("\nTrainee: ").strip().lower()

    # 5. Step-by-step guidance
    steps = parse_steps(recipe_text)
    for idx, step in enumerate(steps):
    # Chef's turn
        chef_step = chef.respond(conversation, f"Step {idx+1}: {step}\nExplain...")
        print(f"\nChef (Step {idx+1}):", chef_step)
        append_with_accrual(conversation, AIMessage(content=chef_step), chef, trainee, combined_accrual)

        # Trainee's turn
        trainee.current_step = idx
        trainee_msg = trainee.generate_response(chef_step, len(steps))
        print("\nTrainee:", trainee_msg)
        append_with_accrual(conversation, HumanMessage(content=trainee_msg), chef, trainee, combined_accrual)

    # Log interaction metrics
    interaction_log.append({
        "step": idx+1,
        "chef_tokens": chef.token_cost_log[-1]["total_tokens"],
        "trainee_tokens": trainee.token_cost_log[-1]["total_tokens"],
        "total_cost": chef.token_cost_log[-1]["cost"] + trainee.token_cost_log[-1]["cost"],
        "cumulative_tokens": chef.token_cost_log[-1]["cumulative_tokens"] + trainee.token_cost_log[-1]["cumulative_tokens"],
        "cumulative_cost": chef.token_cost_log[-1]["cumulative_cost"] + trainee.token_cost_log[-1]["cumulative_cost"]
    })

    # 6. End message
    chef_end = chef.respond(conversation, "Say cooking is complete and offer congratulations.")
    print("\nChef:", chef_end)
    conversation.append(AIMessage(content=chef_end))

    # 7. Save conversation to JSON

    

    conversation_log = []
    for i, m in enumerate(conversation):
        entry = {
            "role": "trainee" if isinstance(m, HumanMessage) else "chef",
            "content": m.content
        }
        if i < len(combined_accrual):
            entry.update(combined_accrual[i])
        conversation_log.append(entry)


    chef_total_tokens = sum(entry["total_tokens"] for entry in chef.token_cost_log)
    chef_total_cost = sum(entry["cost"] for entry in chef.token_cost_log)

    # Aggregate Trainee stats
    trainee_total_tokens = sum(entry["total_tokens"] for entry in trainee.token_cost_log)
    trainee_total_cost = sum(entry["cost"] for entry in trainee.token_cost_log)

    # Overall totals
    overall_total_tokens = chef_total_tokens + trainee_total_tokens
    overall_total_cost = chef_total_cost + trainee_total_cost

    # Print results
    print("\n=== Token and Cost Summary ===")
    print(f"ChefAgent:    {chef_total_tokens} tokens, ${chef_total_cost:.6f}")
    print(f"TraineeAgent: {trainee_total_tokens} tokens, ${trainee_total_cost:.6f}")
    print(f"TOTAL:        {overall_total_tokens} tokens, ${overall_total_cost:.6f}")


    last_chef = chef.token_cost_log[-1]
    print(f"ChefAgent running total: {last_chef['cumulative_tokens']} tokens, ${last_chef['cumulative_cost']:.6f}")

    last_trainee = trainee.token_cost_log[-1]
    print(f"TraineeAgent running total: {last_trainee['cumulative_tokens']} tokens, ${last_trainee['cumulative_cost']:.6f}")

    if 'combined_accrual' not in locals():
        combined_accrual = []

    # Get last chef and trainee cumulative totals (or 0 if none yet)
    chef_cum = chef.token_cost_log[-1] if chef.token_cost_log else {"cumulative_tokens": 0, "cumulative_cost": 0}
    trainee_cum = trainee.token_cost_log[-1] if trainee.token_cost_log else {"cumulative_tokens": 0, "cumulative_cost": 0}

    combined_accrual.append({
        "turn": len(combined_accrual) + 1,
        "chef_cumulative_tokens": chef_cum["cumulative_tokens"],
        "chef_cumulative_cost": chef_cum["cumulative_cost"],
        "trainee_cumulative_tokens": trainee_cum["cumulative_tokens"],
        "trainee_cumulative_cost": trainee_cum["cumulative_cost"],
        "overall_cumulative_tokens": chef_cum["cumulative_tokens"] + trainee_cum["cumulative_tokens"],
        "overall_cumulative_cost": chef_cum["cumulative_cost"] + trainee_cum["cumulative_cost"]
    })


    # Prepare summary for saving
    token_cost_summary = {
        "chef_total_tokens": chef_total_tokens,
        "chef_total_cost": chef_total_cost,
        "trainee_total_tokens": trainee_total_tokens,
        "trainee_total_cost": trainee_total_cost,
        "overall_total_tokens": overall_total_tokens,
        "overall_total_cost": overall_total_cost,
        "chef_turns": chef.token_cost_log,
        "trainee_turns": trainee.token_cost_log
    }

    with open(f"cooking_session_{job_id}_full_log.json", "w", encoding="utf-8") as f:
        json.dump(
            {
                "job_id": job_id,
                "conversation": conversation_log,
                "metrics": {
                    # "interactions": interaction_log,  # New per-interaction data
                    "chef_turns": chef.token_cost_log,
                    "trainee_turns": trainee.token_cost_log,
                }
            },
            f, indent=2, ensure_ascii=False
        )


# Usage with your recipe
if __name__ == "__main__":
    job_id = str(uuid.uuid4())
    trainee_experience_log = {
        "experience_level": "Beginner",
        "preferred_cuisine": "Italian",
        "allergies": ["milk"],
        "notes": "Has never roasted a chicken"
    }
    with tracing_v2_enabled(project_name="run-traces", tags=[f"job_id:{job_id}"]):
        automated_cooking_session(job_id, trainee_experience_log=trainee_experience_log)
